# Lecture 13 — Visualization, and When Data Isn't Normal

**Part 1** builds up the plotting toolkit on the heart-disease dataset: histograms,
comparing populations, 1-D, 2-D and 3-D plots.
**Part 2** uses that toolkit for a question you will hit in every dataset you ever
histogram: *the Central Limit Theorem says things should look normal — so why doesn't
this?* **Part 3** is basic data manipulation on the workplace-fatalities data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Part 1 — Visualization

### Heart Data

[Source](https://www.kaggle.com/ronitf/heart-disease-uci/version/1#)

* *age*- age in years
* *sex*- (1 = male; 0 = female)
* *cp*- chest pain type
* *trestbps*- resting blood pressure (in mm Hg on admission to the hospital)
* *chol*- serum cholestoral in mg/dl
* *fbs*- (fasting blood sugar > 120 mg/dl) (1 = true; 0 = false)
* *restecg*- resting electrocardiographic results
* *thalach*- maximum heart rate achieved
* *exang*- exercise induced angina (1 = yes; 0 = no)
* *oldpeak*- ST depression induced by exercise relative to rest
* *slope*- the slope of the peak exercise ST segment
* *ca*- number of major vessels (0-3) colored by flourosopy
* *thal*- 3 = normal; 6 = fixed defect; 7 = reversable defect
* *target*- 1 or 0

The list above is the documentation that came with the file. A dataset normally arrives in two
pieces like this: a table, and a separate explanation of what its columns mean. Here several entries
are opaque even to a reader who knows the field, and working without resolving them is deliberate —
deducing what a variable is from how it is distributed is the position a new dataset normally puts
you in. Where that is not enough, the next step is to ask someone who works in the field; the more
domain knowledge is brought to a dataset, the less likely a column is to be misread.

This particular dataset is small, widely circulated and inconclusive: roughly three hundred patients,
and it does not have the statistical power to settle the question it was built for. The obvious
expectation — that patients with heart trouble carry higher cholesterol — is not visible in these
rows. A dataset failing to show an effect is not evidence that the effect does not exist. It earns its place here as a worked example,
not as something to build a project on.

In [ ]:
!head heart.csv

Looking at the raw lines before loading anything answers two questions. First, whether the file will
load at all: the fields are comma separated, there is a single header row and there is no stray
text, so `read_csv` will handle it without repair. Second, which columns are categorical and which
are numerical — `sex` is zeros and ones, `cp` runs over a few small integers, while `chol` and
`trestbps` take a wide spread of values. That split decides which kind of display each column gets
for the rest of the chapter. `head` prints the first ten lines of a file and `tail` the last ten,
which is usually enough to tell.

Lets investigate how to visualize it.

In [ ]:
df = pd.read_csv("heart.csv")

In [ ]:
df

Three hundred and three patients, fourteen columns. A common next move is a table of summary
statistics; this walkthrough goes to the features instead. What the printed table does show is that
the rows are not in random order — the first rows all have `target` equal to one and the last rows
all have zero. The file is sorted by outcome, so the first few and last few rows are not a fair
sample of it, and anything that depends on row order has to shuffle first. Part 2 returns to this
with a plot that detects it.

## Matplotlib

`matplotlib` is the most commonly used plotting tool in the python ecosystem. There is extensive documentation, and you are encouraged to look through the [Tutorials](https://matplotlib.org/3.3.3/tutorials/index.html).

Other packages sit on top of `matplotlib` rather than replacing it. Seaborn packages common
statistical plots into single calls, and Plotly aims at interactive dashboards, in the spirit of
Tableau. The cost of a single call is that it makes choices on your behalf, and a choice you did
not make is one you may not notice, which is how a figure comes to be misread. That is the reason
for building a histogram from scratch earlier in the course: a plot you could have written yourself
is a plot you can interpret. Work through the tutorials above once — not to memorise the calls, but
to learn what the library can do, so that later the thing to look up has a name.

But before we dive into the techinical aspects of making plots, let's first think of what type of plots may be useful in a data science context.

It helps to separate three levels of data. **Raw data** is images, sound, video, or the measurements
an instrument records at each event; there is little to visualize beyond opening one up and checking
that it looks like what it claims to be. **Tabular data** is one row per data point and one column
per random variable — patient records, transactions, grades — and this is the level every plot in
Part 1 addresses. **Summary statistics** are the output of an analysis of a table: an average
graduation rate per major, a fatality rate per state. Raw data becomes tabular data by processing:
run algorithms over the images, extract numbers such as whether a cat appears, and put those numbers
in a table. Most charts in general circulation live at the third level. Part 3 returns to it.

## Histograms

As discussed throughout the course, a dataset typically consists of data points, each of which is a set of observations of the same set of random variables. What we are describing is a table, with each row corresponding to a data point, and each column a random variable. 

The reason to reach for a histogram first is that every column is a random variable, and a
histogram is the estimate of that variable's probability distribution that a finite sample allows.
If a distribution were known exactly, in functional form, there would be nothing further to learn
from the data — it would already contain everything there is to know. Where a dataset holds two
populations, each has its own distribution for each variable, and asking whether a variable carries
information about which population a patient belongs to is asking whether those two distributions
differ. That is also the fastest first answer to a new problem: handed a labelled dataset and asked
whether it can separate fraud from not-fraud, histogram every column for both populations and look
for one where the two curves pull apart. A column whose two distributions sit on top of each other
carries no information about the label.

Lets look at one variable:

In [ ]:
plt.hist(df["chol"])

The call returns three things: the array of counts, the array of bin edges — one longer than the
counts, since N bins have N+1 edges — and a container holding the objects that were drawn. Those
objects are the bars themselves, each one a thing in memory that has been told to draw itself. They
are handed back so that a reference to them survives: without one, Python would collect them, and
anything that later needed to redraw the figure would have nothing left to redraw. Keeping the
reference is also what lets a plot be adjusted after the fact, by reaching into the container and
changing an object. That is not the workflow here — these plots are made and left alone — but it is
why the call returns them.

Read what the first histogram says: cholesterol centres somewhere around 240, with a long right
tail and a few patients far out. Beyond that, the picture is a choice. The plot has ten bins because
ten is `plt.hist`'s default, not because ten suits this variable. A bin is a width over which the
data is averaged, so bins that are too wide smear structure away — two nearby bumps falling inside
one bin arrive as a single bump, and nothing in the picture hints that anything was lost. Whenever
a histogram looks featureless, the bin width is the first thing to vary.

with more bins:

In [ ]:
plt.hist(df["chol"],bins=50)

At fifty bins there are bumps, and a bump is not yet a finding. Each one has three kinds of
explanation. It can be a data error: one patient entered ten times would fill a bin by itself. It
can be a real sub-population, some group of patients who share a value for a reason. Or it can be
noise, and deciding that is arithmetic. Filling a histogram is a counting experiment, so a bin
holding $n$ entries carries an uncertainty of about $\sqrt{n}$. If a smooth distribution would put
ten entries in a bin, then roughly two thirds of the time the count lands within $\sqrt{10}$ of
that — about three either side — so a bin holding fourteen where ten was expected is a little high
and nothing more, and a count like that will turn up in a handful of bins out of fifty by chance
alone. Before reading a bump as a sub-population, compare its height against the square root of its
own contents. At this sample size the honest conclusion is that these bumps are consistent with
fluctuation, and that a real effect of the same size would be invisible for the same reason.

`plt.hist` returns three things, and a notebook echoes the value of the last expression in a cell,
so calling it bare prints all three above the figure. Assigning the result to `_` stops that. The
underscore is not special syntax: it is an ordinary variable name, used by convention for a value
that is being thrown away.

Suppress the return values:

In [ ]:
_=plt.hist(df["chol"],bins=50)

The next two cells draw the same column over a range of bin counts, from five to five hundred, which
shows both failure modes at once. Normalized and laid on top of one another they agree: they are
several views of one distribution, and all of them carry exactly the same statistical power, because
they come from the same three hundred numbers. Changing the bin count does not add or destroy
information. It trades resolution against the certainty of each bin.

In [ ]:
for bins in [5,10,25,50,100,500][::-1]:
    print("Number of bins:",bins)
    _=plt.hist(df["chol"],bins=bins,density=1,label=str(bins), alpha=0.5)
    #plt.show()
plt.legend()

In [ ]:
for bins in [5,10,25,50,100,500]:
    print("Number of bins:",bins)
    _=plt.hist(df["chol"],bins=bins)
    plt.show()
#plt.legend()

Each extreme costs something specific. At five hundred bins most bins are empty — 348 of the
500 — and the ones that are not hold between one and six patients, since a bin with an expectation
above zero still has a real chance of coming up empty. A bin holding four is $4 \pm 2$, so it could
as easily have been zero or seven. Every apparent
feature at that resolution is a candidate fluctuation. Wide bins have the opposite problem: a bin holding sixteen
entries would be $16 \pm 4$, a quarter of the value, which is a usable uncertainty, but by then
neighbouring regions of the distribution have been averaged together and a difference between them
cannot be seen. The bin count is chosen between those: as few bins as the question allows, but
enough counts per bin that two parts of the distribution can be told apart. How many that is depends
on how much data there is, not on the variable — as the sample grows, the affordable resolution grows with it.

There is no right answer to the bin count, which is exactly why the plot has to be looked at. There
is a second
reason to care: a statistical test can find a feature that the eye cannot, but a plot that does not
show the feature will not convince anyone else of it, so part of choosing a binning is choosing a
plot that carries the argument. A histogram made to show someone else is a different plot from one
made while hunting — fewer bins, and error bars, so a reader can see which differences the sample
size supports. The failure this guards against is routine: code runs, a figure appears, the figure
is handed on, and nobody has read it. Bringing a plot you know is broken and cannot explain is a
normal part of analysis; bringing one that is broken without having noticed is not.

A worked example makes the trade-off concrete, and it is the case this chapter cares about most.
Suppose cholesterol among the patients without heart disease is normal, and among the patients with
it the same shape shifted higher — here by two and a half standard deviations. A measurement does
not arrive with a label attached, so a histogram of everyone is the sum of the two, and the question
is whether that sum can be told apart from a single population. Whether it can depends on two things
at once: how many patients there are, and how finely the histogram is binned. The next section runs
exactly this comparison on the real cholesterol column, where the two populations turn out to be
nothing like this far apart.

In [ ]:
np.random.seed(3402)

n_patients=[10,100,1000,10000]
bin_counts=[10,25,50,100,500]

plt.figure(figsize=(16,12))

plot_i=0
for n in n_patients:
    sample=list(np.random.normal(0.,1.,n//2))+list(np.random.normal(2.5,1.,n//2))

    for n_bins in bin_counts:
        plot_i+=1
        plt.subplot(len(n_patients),len(bin_counts),plot_i)
        _=plt.hist(sample,bins=n_bins,range=(-5.,7.5))
        plt.title(str(n)+" patients, "+str(n_bins)+" bins",fontsize=9)
        plt.xticks([])
        plt.yticks([])

plt.tight_layout()
plt.show()

Read the grid as a map of when the second population is visible at all. With ten patients nothing is
visible at any binning: every panel is a handful of bars in arbitrary places. With a hundred there
is sometimes a shoulder, and whether it shows depends on the binning more than on the data. From a
thousand upwards the two humps are plain over a wide middle range of bin counts, and the limit is
the one described above — more data buys finer bins, and the fine binnings that were pure noise at a
hundred patients start to carry the shape at ten thousand. The same two humps are present in
every panel of a row, since each row is one sample drawn once; what changes across a row is only
whether the binning lets them be seen.

Two normal populations this far apart are close to the easiest case. Part 2 takes up the general
question — how far apart two populations must be before a histogram shows two humps rather than one
lopsided one — and the answer there is about two standard deviations, just below the separation
used here.

## Comparing Populations

The eventual goal is a model that takes a patient's measurements and returns the probability that
they have the disease. This dataset is labelled, so the populations are known in advance and the
question is narrower: for each feature, does the distribution differ between the two groups? A
feature whose distributions differ can be used to tell the groups apart. A feature whose
distributions coincide cannot, however well it is plotted.

There are two populations in the dataset, those with and without heart disease. We would like to see if any of features about the patients (e.g. their cholestrol level) correlate with the disease. Let's first divide us the data into the two populations:

In [ ]:
disease_selection=df["target"]==1
no_disease_selection=df["target"]==0

A comparison written against a column does not return one value: it returns one answer per row, a
column of `True` and `False` the same length as the frame. Such a column is a mask, and indexing the
frame with it gives back only the rows where it is `True`. Building two separate frames up front
would work equally well; keeping the masks instead means the frame is never copied, and either
population can be named wherever it is needed.

The two populations here are the patients with and without heart disease, which is the `target`
column, and Part 2 splits on the same column. Nothing below depends on that choice: any pair of
masks would work, and the four-way comparison a few cells on splits by chest-pain type instead.
What does matter is that the populations compared are the ones the question is about — `cp`, the
chest-pain type, is a symptom rather than an outcome, so a comparison by it answers a different
question — which is what the four-way comparison a few cells on is doing.

In [ ]:
disease_selection

Then separately historgram the feature to see if the distributions look different for the two populations:

In [ ]:
_=plt.hist(df[disease_selection]["chol"],bins=50)
_=plt.hist(df[no_disease_selection]["chol"],bins=50)

There is a subtler problem here than the missing legend. Asking for fifty bins does not say where
the bins go: the rule is the one used in the from-scratch histogram exercise — take the minimum and
maximum of the data supplied, and divide that range into fifty equal parts. The two populations
have different minima and maxima, so each call chooses its own edges, and the two sets of bars end
up offset from each other by a fraction of a bin. Two histograms drawn on slightly different grids
are easy to misread as two distributions that differ. The fix appears in the next cell: keep the bin
edges the first call returns, and pass them to every later call.

Hmmm... which is which? Need a legend:

In [ ]:
_,bins,_=plt.hist(df[disease_selection]["chol"],bins=50,label="Disease")
_=plt.hist(df[no_disease_selection]["chol"],bins=bins,label="No Disease")
_=plt.legend()

Can't see too well... make the histograms partially transparent:

In [ ]:
_,bins,_=plt.hist(df[disease_selection]["chol"],bins=50,alpha=.5,color='b',label="Disease")
_=plt.hist(df[no_disease_selection]["chol"],bins=bins,alpha=.5,color='r',label="No Disease")
_=plt.legend()

The problem is easiest to see in the extreme. Suppose one population had ten times as many patients
as the other. One histogram would tower over the other even if the two underlying distributions
were identical, because the height of a bin counts patients and one population simply has more of
them. Any difference in height read off such a plot is a difference in population size, not a
difference in distribution.

Not sure if there is a difference in the distributions because there different number of patients in the two populations:

In [ ]:
print(np.sum(disease_selection))
print(np.sum(no_disease_selection))

Here the two populations are 165 and 138 patients — close enough that the distortion is small, and
still worth removing, because the point of the comparison is to decide something and a comparison
with a known bias in it cannot be trusted at the margin. The correction is a division: each bin's
count over the total number of entries, and over the bin width, so that the area under the
histogram is one rather than the number of patients. The six binnings compared earlier relied on the
same correction; what made those histograms lie on top of one another, despite wildly different
counts per bin, was that each was normalized to unit area. Counts are not comparable across samples
of different size. Densities are.

What we need is to "normalize" the histogram so the integral is the same, namely 1, therefore we can interpret the histogram as a density:

In [ ]:
_,bins,_=plt.hist(df[disease_selection]["chol"],bins=50,alpha=.5,color='b',density=1,label="Disease")
_=plt.hist(df[no_disease_selection]["chol"],bins=bins,alpha=.5,color='r',density=1,label="No Disease")
_=plt.legend()

Four changes have accumulated, and each fixed one defect: labels and a legend so the populations can
be told apart, transparency so the one drawn second does not hide the first, shared bin edges so the
bars line up, and a density normalization so the areas are comparable. The result is readable, and
what it shows is close to nothing. The two distributions sit on top of each other. The group means
differ by about nine mg/dl, in the direction opposite to the expectation the comparison was built to
test, but that is well inside the uncertainty on a sample this size and is not something this plot
can be read as showing. That is the result, and it is worth stating plainly: on this feature, the honest report is that there is probably nothing here to build
on. Reporting a negative result from a plot like this is as much of an answer as reporting a positive
one.

What this pair of normalized histograms does by hand is the shape of a problem called binary
classification: two populations, a set of features measured on both, a labelled sample to learn from
and an unlabelled one to predict on. A classifier is characterising the probability distribution of
each population over the features — the same thing this plot does for one feature at a time, read by
eye.

The sequence above took five cells to get right, and it applies to exactly one column. There are
fourteen. Wrapping it in a function is not housekeeping: the cost of asking a question decides
whether the question gets asked at all. A plot that takes ten minutes to set up is a plot that does
not get made, and the hypothesis behind it goes unchecked. A plot that takes one line gets made on
a hunch.

Now we want to do it for all the features, so lets write a function, first to nicely compare distributions of different populations:

Note that every population is histogrammed with the **same bin edges** — the first call
returns the edges it chose and the later calls reuse them. Without that, overlaid
histograms of different populations get different bins and are hard to compare.

In [ ]:
def compare_distributions(df,column_name,selections,bins=10,**kwargs):
    for label,selection in selections.items():
        _,bins,_=plt.hist(df[selection][column_name],bins=bins,label=label,**kwargs)

    _=plt.legend()

`**kwargs` in the signature is what makes this function usable. `plt.hist` accepts dozens of keyword
arguments, and naming them all here would mean editing the function every time a new one is needed.
Instead `**kwargs` collects every keyword argument the caller passed that the function did not name,
into a dictionary; writing `**kwargs` at the call site unpacks that dictionary back into keyword
arguments for `plt.hist`. The effect is a pass-through: `alpha`, `density` and `color` are chosen by
the caller and handled by `plt.hist`, and nothing in between needs to know they exist. Library code
uses the same mechanism throughout, which is how a setting can reach a function several layers below
the one being called.

Passing the populations as a dictionary means the keys are the legend labels and the values are the
masks, so changing what is being compared is a matter of editing a dictionary literal rather than
the plotting code. Switching what is being compared therefore means editing a dictionary literal and nothing else: the
four-way chest-pain comparison in the next cell is the same function call with a different
dictionary. One thing
is lost in the move to a function — the hand-built version pinned `color='b'` and `color='r'`, while
the function lets `matplotlib` assign colours from its default cycle, so the colours here will not
match the earlier plots.

Note that we define the populations using a dictionary:

In [ ]:
compare_distributions(df,"chol",
                     {"Disease":df["target"]==1,
                      "No Disease":df["target"]==0},
                     alpha=0.5,
                     density=1,
                     bins=25,
                     )

In [ ]:
compare_distributions(df,"chol",
                     {"CP 0":df["cp"]==0.,
                      "CP 1":df["cp"]==1.,
                      "CP 2":df["cp"]==2.,
                      "CP 3":df["cp"]==3.,
                      #"CP Not 0":df["cp"]!=0
                     },
                     alpha=0.5,
                     density=1,
                     bins=25,
                     )

This comparison splits the data a different way — not by outcome but by chest-pain type — which is
a reasonable thing to ask of the same function, and it exposes the cost of normalizing. Because each
curve is scaled to unit area, a category with very few patients is drawn just as tall as one with many, and its sample size
is no longer visible in the height. It is still visible in the shape: chest-pain type 3 covers only
23 of the 303 patients, so spread over twenty-five bins its bins hold one or two patients each and
it appears as a comb of spikes rather than a distribution. A histogram being correctly normalized
and a histogram being readable are different things.

With the comparison in a function, the whole table can go through it. Doing that — every feature,
both populations, shared bins, normalized — is the first thing to do with a new dataset, before any
modelling and before choosing which features to care about. `plt.show()` inside the loop is what
makes this fourteen figures rather than one; without it every call draws onto the same axes and the
whole dataset lands in a single unreadable plot. One of the fourteen panels is not a result but a
by-product. The `target` panel plots the column the populations were defined by, so within each
population that column holds a single value; and since `compare_distributions` reuses the bin edges
the first call chose, those edges span only the first population's one value and the second
population falls outside them and is not drawn at all. The panel is a standing reminder that shared
bin edges are shared with whatever range the first call happened to pick.

Now lets histogram every feature:

In [ ]:
selection_dict={"Disease":df["target"]==1,"No Disease":df["target"]==0}

for column_name in df.columns:
    print(column_name)
    compare_distributions(df,column_name,
                     selection_dict,
                     alpha=0.5,
                     density=1,
                     bins=25,
                     )
    plt.show()

Reading a pair of overlaid distributions comes down to a few recognisable shapes, and what is being
looked for in a wall of histograms is any of them, in any column.

* **Two separated peaks.** A single threshold separates the populations: a patient on one side of it
  probably belongs to one population, on the other side to the other.
* **Two curves on top of each other.** The variable carries no information about which population a
  patient is in.
* **Same centre, different widths.** No threshold works, but a band does — the useful rule is on the
  distance from the mean, whether a value lies within one standard deviation of it, say.
* **Same shape, shifted centres.** A threshold works, with the sense of the comparison reversed for
  the two populations.

There is a fifth shape, and it is the one that starts an investigation. A single population's
histogram can itself be the sum of several sub-populations, and what the plot shows is the sum, not
the parts; a bump or a shoulder is the visible trace of one of them. The move is then to select the
patients in that bump and histogram their *other* columns. If they are a real group, something else
about them will differ too, and if nothing does, the bump was a fluctuation. Part 2 takes this up as
a named failure of the assumption that one histogram describes one population.

In [ ]:
len(df.columns)

Fourteen separate figures have to be scrolled, and two panels that cannot be seen at the same time
cannot be compared — which is why the column count is worth asking for. Fourteen variables fit a
five-by-three grid with one cell to spare: `plt.figure(figsize=(15,15))` sets the size of the whole
canvas, and `plt.subplot(5,3,i+1)` selects which of the fifteen cells the next plot goes into,
counting across rows from the top left. Packing the panels into one figure is not cosmetic. Fourteen
is also a small number of features — thousands is ordinary — and making one plot, looking at it,
discarding it and making the next throws away most of what the plots could have said.

In [ ]:
selection_dict={"Disease":df["target"]==1,"No Disease":df["target"]==0}

plt.figure(figsize=(15,15))

for i,column_name in enumerate(df.columns):
    plt.subplot(5,3,i+1)
    compare_distributions(df,column_name,
                     selection_dict,
                     alpha=0.5,
                     density=1,
                     bins=25,
                     stacked=True
                     )
    plt.xlabel(column_name)
plt.show()

Read as a whole, the grid says where to look. The routine clinical numbers barely differ between the
two groups: cholesterol, resting blood pressure and fasting blood sugar put the two distributions
almost on top of each other. The exercise-test results do not. Maximum heart rate (`thalach`), the
ST depression (`oldpeak`), exercise-induced angina (`exang`) and the slope of the ST segment
separate visibly, as do chest-pain type (`cp`) and the number of vessels seen under fluoroscopy
(`ca`). Sex separates too, and age shifts a little, the patients with the disease being the younger
group. The conclusion
from a few minutes of histogramming, with no analysis at all, is that whatever separating power this
dataset has lives in the test results rather than in the routine measurements.

A histogram of a column that takes three or four values holds only three or four numbers, so drawing
it as bars adds nothing and makes those numbers harder to read off. Worse, several of these features
are drawn with twenty-five bins, which turns a two-valued variable into a pair of isolated spikes
with empty space between them; if it is to be a histogram at all it needs two bins. A table loses
nothing and is easier to compare across populations, and it has room for something the histogram
does not: the counting error beside each fraction, which is what decides whether a difference
between two populations is real or is the sample size talking.

For the categorical features, the information in the histogram can be summarized better in a table of what fraction of each population belongs to each category. Quick example of such tables:

In [ ]:
def compare_categorical(df,column_name,selections,**kwargs):
    out = list()
    values = df[column_name].unique()
    
    for label,selection in selections.items():        
        total = np.sum(selection)
        df_0=df[selection]
        out.append([label] + [ np.sum(df_0[column_name]==val)/total for val in values ])
        
    return out,values

For each population in turn, the function finds the distinct values the column takes, counts how
many members of that population have each value, and divides by the size of that population. Each
row of the result is therefore a set of fractions that add to one: of the patients in this
population, this fraction had each value. One detail of the bookkeeping is worth seeing, because getting it wrong is easy and the result still
looks like a table. The distinct values are collected once, from the whole frame, before the loop.
Collecting them inside the loop instead would give each population the values in whatever order that
population happens to list them, while the table prints a single shared header — so for any column
whose two populations differ in that order, one row would be printed under the other's headings, and
the table would read as the exact reverse of the truth for that population.

In [ ]:
from IPython.display import HTML, display
import tabulate

`tabulate` turns a list of rows into a formatted table. Asked for `tablefmt='html'` it returns HTML
rather than plain text, and `IPython.display`'s `HTML` and `display` render that HTML in the
notebook as a real table instead of printing the markup. The same function with its default format
prints a plain-text table, which is what is wanted outside a notebook.

In [ ]:
results = compare_categorical(df,"sex",selection_dict)
print(results)
display(HTML(tabulate.tabulate(results[0], tablefmt='html', headers=results[1])))

In [ ]:
for cat_feature in ["sex","thal","ca", "slope", "exang","restecg", "fbs"]:
    print(cat_feature)
    results = compare_categorical(df,cat_feature,selection_dict)
    display(HTML(tabulate.tabulate(results[0], tablefmt='html', headers=results[1])))
  

These tables answer the wrong question. Dividing by the size of each population gives the fraction of
that population having each value — in the notation of the probability chapter,
$P(x \mid \text{population})$. The question a diagnosis asks runs the other way: given a patient with
this value, which population do they belong to, $P(\text{population} \mid x)$? Those are different
numbers, and one cannot be read off the other without also knowing how large each population is. The
difference is not cosmetic. Of the patients without the disease, 83% are male; of those with it, 56%
are. Read the other way, three quarters of the women in this sample have the disease, against 45% of
the men. Both numbers are real, and only the second is a statement about a patient in front of you —
and even that one inherits the composition of the sample, so it is not a prevalence in the world. The next cell normalizes down the columns instead
of across the rows — for each value of the column, the fraction of all patients with that value who
fall in each population. That is the number that can be used on a new patient.

In [ ]:
def compare_categorical_T(df,column_name,selections,**kwargs):
    out = list()
    values = df[column_name].unique()        
    N_val = dict([ (val, np.sum(df[column_name]==val)) for val in values ])

    for label,selection in selections.items():        
        total = np.sum(selection)
        df_0=df[selection]
        out.append([label] + [ np.sum(df_0[column_name]==val)/N_val[val] for val in values ])       
    
        
    return out,values


for cat_feature in ["sex","thal","ca", "slope", "exang","restecg", "fbs"]:
    print(cat_feature)
    results = compare_categorical_T(df,cat_feature,selection_dict)
    #print(results[0])
    display(HTML(tabulate.tabulate(results[0], tablefmt='html', headers=results[1])))
  

Now the tables can be read as statements about a patient. Where a row is close to even between the
populations, that value says nothing. Where it is lopsided — two thirds one way, say — knowing a
patient's value shifts the odds by that much. The same information was present in the histograms and
could not be read off them. And these numbers compose: one probability per column, combined across
columns, is a procedure for taking a new patient's measurements and returning a probability that
they belong to one population rather than the other, which is the simplest form of the classifiers
the machine-learning part of the course builds. Neither table carries an uncertainty, which is the
next thing they need: a fraction like two thirds computed from a handful of patients cannot be
distinguished from the same fraction computed from hundreds, and the $\sqrt{n}$ on each count
carries through to the fraction.

## 1-D Plots

Everything so far has been a histogram, which is in a sense a zero-dimensional plot: nothing is
plotted against anything, a single variable is counted and the other axis holds only that count. A
bar graph is the same picture with the bars separated and labelled, which suits a variable whose
values are names rather than numbers. A genuine x-against-y plot is a different object, and its
natural subject is a function — build an array of x values, evaluate the function at each one, and
draw the result, as the next cells do for sine and cosine. A probability density is such a function,
with the extra properties that it cannot be negative and that it integrates to one, so it can be
drawn this way; the most common use for a 1-D plot here is to overlay a fitted density on a
histogram. A column of data is not a function — there is no single y for a given x — which the third
cell below demonstrates.

In [ ]:
import math

x = np.linspace(-2.*math.pi,2.*math.pi,100)
y = list(map(math.sin,x))

_=plt.plot(x,y)

In [ ]:
x = np.linspace(-2.*math.pi,2.*math.pi,100)
y_1 = list(map(math.sin,x))
y_2 = list(map(math.cos,x))

_=plt.plot(x,y_1,label="sin")
_=plt.plot(x,y_2,label="cos")

_=plt.legend()

There is one x-against-y plot that does belong to data work. Training a model means repeatedly
adjusting it to reduce an objective, and plotting that objective against the iteration number gives
a curve that falls and then flattens. It is the plot that shows whether training ran at all, whether
it had finished, and whether it went wrong — which is why a reported training result without its
curve should not be believed, including by the person who produced it.

In [ ]:
df.columns

In [ ]:
_=plt.plot(df["thalach"],df["chol"])

That plot is deliberately wrong. `plt.plot` connects consecutive points with straight lines, which
assumes the points arrive in a meaningful order and that each x has exactly one y. Neither holds for
a table of patients: the order is whatever order the rows happen to be in, and many patients share a
maximum heart rate while having different cholesterol. The result is a scribble in which every line
segment is an artefact of row order. Sorting the rows first can make such a plot readable for a
particular purpose, but two columns of data are not plotted against each other this way.

## 2-D Plots

When in comes to data, looking at 2 random variables together can serve to see trends in correlations between the variables. 2D histograms are one way to see this info 

A two-dimensional histogram is the same construction one dimension up: bin both variables, then
count how many data points fall in each box. Displaying it is the new problem. Both directions on
the page are already spent on the two variables, so the count needs a third channel, and colour is
it — bright where many patients fall, dark where few do. A plot of that form is a heat map, and
reading one means reading brightness as density. What would make such a plot valuable is two
populations occupying different regions, because then a line drawn across the plane separates them;
finding that line, in many more than two dimensions, is what the machine-learning part of the course
automates.

In [ ]:
_=plt.hist2d(df["thalach"],df["chol"])

But they can be difficult to compare trends in correlations in different populations.

The difficulty is more general than comparing populations. Two colour maps cannot be overlaid the
way two histograms can — one would simply cover the other — so comparing populations this way means
two separate figures and flipping between them, which is the comparison the eye is worst at. Sample
size makes it worse: about 150 patients spread over a ten-by-ten grid leaves half the cells empty and
most of the rest holding a handful, so much of the colour variation is counting noise. Two responses follow — the scatter plot below, and
the contour plot at the end of this section.

In [ ]:
selection_dict={"Disease":df["target"]==1,"No Disease":df["target"]==0}

for label,selection in selection_dict.items():   
    _=plt.hist2d(df[selection]["thalach"],df[selection]["chol"],label=label)

    _=plt.xlabel("thalach")
    _=plt.ylabel("chol")

    plt.show()

A scatter plot draws one marker per data point at its two coordinates — here, one dot per patient —
and joins nothing, which removes the artefact that made the line plot meaningless. The
two-dimensional histogram is this same plot with the plane cut into boxes and the dots in each box
counted, so the histogram adds nothing and throws the exact positions away. At this sample size the
dots are few enough to see individually and the eye does the counting on its own: a region with more
dots in it reads as denser without any need to bin.

Scatter plots can be useful in this instance:

In [ ]:
_=plt.scatter(df["thalach"],df["chol"])

In [ ]:
selection_dict={"Disease":df["target"]==1,"No Disease":df["target"]==0}

for label,selection in selection_dict.items():   
    _=plt.scatter(df[selection]["thalach"],df[selection]["chol"],label=label)

_=plt.xlabel("thalach")
_=plt.ylabel("chol")

_=plt.legend()

The advantage over the two-dimensional histograms is that both populations fit on one set of axes,
in two colours, with a legend. The question to ask of such a plot is whether the two colours occupy
different regions, because a boundary drawn between them would then separate the populations using
both variables at once — something neither variable alone could do. That is the second way a dataset
can let two populations be told apart: not a single feature whose distributions differ, which the
histograms were looking for, but a relationship between two features that differs between the
populations, each group following its own trend. Here the colours are offset along the
maximum-heart-rate axis and overlap heavily, and that offset is the one-dimensional difference
already visible in that variable's own histogram — the pair adds little beyond it.

In [ ]:
selection_dict={"Disease":df["target"]==1,"No Disease":df["target"]==0}

def compare_scatter(df,x_var_name,y_var_name,selections,make_legend=True) :
    for label,selection in selections.items():   
        _=plt.scatter(df[selection][x_var_name],df[selection][y_var_name],label=label)

    _=plt.xlabel(x_var_name)
    _=plt.ylabel(y_var_name)

    if make_legend:
        _=plt.legend()

In [ ]:
compare_scatter(df,"thalach","chol",selection_dict)

Every variable against every other is 14 × 14 = 196 plots — too many to read. Here are the
first five columns against each other; the next cells pack these into a single grid.

In [ ]:
for x_var_name in df.columns[:5]:
    for y_var_name in df.columns[:5]:
        if x_var_name != y_var_name:
            compare_scatter(df,x_var_name,y_var_name,selection_dict)
            plt.show()

A wall of scatter plots asks two separate questions, and it is worth keeping them apart. The first is
whether the two variables are related at all — whether the cloud tilts, so that one tends to rise as
the other falls. The second is whether the two populations differ, which is whether the two colours
sit in different parts of the cloud. This is also where a two-dimensional view can earn
its place: a pair of variables can carry structure that neither of them shows on its own, visible as
two tilted clouds offset from each other, and the only way to find out is to look. Anything found
that way is a lead rather than a conclusion, since nothing here establishes significance. Feeding the columns to a
statistical procedure and reporting the number it returns skips the step where the structure becomes
visible, and a visible effect is more convincing to a reader than a number.

In [ ]:
columns = df.columns[1:6]
n_columns=len(columns)
plt.figure(figsize=(15,15))

plot_i=0
for i,x_var_name in enumerate(columns):
    for j,y_var_name in enumerate(columns):
        plot_i+=1
        plt.subplot(n_columns,n_columns,plot_i)
        make_legend = plot_i==1
        compare_scatter(df,x_var_name,y_var_name,selection_dict,make_legend=make_legend)

The grid is laid out so that the panel in row *i*, column *j* plots column *i* horizontally against
column *j* vertically. That makes each panel above the diagonal the mirror image of the one below
it, reflected about the diagonal, so half the grid is redundant — and the diagonal itself is worse
than redundant, since each of those panels plots a variable against itself and is always a straight
line. Of twenty-five panels, ten carry information.

## Pair Plot

Since the scatter plot of a random variable against itself is trivial, we can replace the diagonal plots above with the 1D histograms

The result is a pair plot: every pair of features as a scatter plot split by population, with each
feature's own histogram along the diagonal, which is the one panel a scatter plot cannot use. Only a
subset of the columns goes in, because the panels shrink as the grid grows.

In [ ]:
# Pair plot

columns = df.columns[1:6]
n_columns=len(columns)
plt.figure(figsize=(15,15))

plot_i=0
for i,x_var_name in enumerate(columns):
    for j,y_var_name in enumerate(columns):
        plot_i+=1
        plt.subplot(n_columns,n_columns,plot_i)
        make_legend = plot_i==1
        if i==j:
            compare_distributions(df,x_var_name,
                     selection_dict,
                     alpha=0.5,
                     density=1,
                     bins=50,
                     )
        else:
            compare_scatter(df,x_var_name,y_var_name,selection_dict,make_legend=make_legend)

A pair plot puts that wasted diagonal to work: where the row and column indices match, draw the
single-variable histogram instead of a scatter plot of a variable against itself, which is what the
`if i==j` branch does. The result is one figure carrying every variable's distribution for both
populations and every pairwise relationship between them. On an unfamiliar dataset this is the plot to
make first. It will also show things that are not there, so what it produces is a list of leads
rather than conclusions.

Seabord makes pairplots easy:

In [ ]:
import seaborn as sns

In [ ]:
sns.pairplot(df,hue="target")

The convenience comes at a price, and it is worth naming. Compare this diagonal against the
hand-built one above: it is not a histogram but a smoothed density estimate, and smoothing makes
every variable look like a tidy bell curve whether or not it is one — the comb of spikes seen
earlier would come out of it as a clean hump. Everything this section used histograms for (sparse
bins, low-statistics spikes, whether a bump survives its own uncertainty) is invisible in a smoothed
curve. It also draws the categorical columns as though they were continuous, and it is slow on a
frame this wide. Smoothing a distribution is a legitimate thing to do, but it is a modelling
decision, and it should be made deliberately with a model chosen for the data rather than inherited
from a default. At the stage of first looking at a dataset, stay as close to the raw counts as
possible: a package that produces a good-looking figure is not a package that has understood the
data.

### How about 3D?

For datasets of random variables, the usefulness of 3D plots are often similar to 1D plots above... not very useful. There are some instances where 3D plots are likely to show up in data analysis:

* 3D visualizations of 2D histograms. Lets quickly look at [example for matplotlib](https://matplotlib.org/stable/gallery/mplot3d/hist3d.html)
* 3D Scatter plots.
* Contour plots. Lets quickly look at [matplotlib contour](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.contour.html).

Of these, only the last two are worth much. A three-dimensional scatter plot makes sense when the
three variables really are positions in space — a point cloud, where the question is which part of
the volume the points fill — but it is readable only if it can be rotated, since any single viewing
angle hides points behind other points, and that needs an interactive package rather than a static
image. A three-dimensional histogram, drawn as a field of bars, is close to unreadable: the bars
occlude each other, and reading a value off one means projecting it back onto an axis by eye. We can
reason about three dimensions and not about four. Of the three, the one actually met often is the
contour plot, which turns up throughout machine learning.

In [ ]:
z,x,y,_=plt.hist2d(df["thalach"],df["chol"])

In [ ]:
plt.contour(y[:-1],x[:-1],np.array(z))

A contour plot is the third way of drawing the same two-dimensional histogram. Drawn as a surface it
would be a landscape, with the counts as heights — a hill where patients are dense, flat ground
where they are not — and a contour plot is the topographic map of that landscape. Each line joins
points of equal value, exactly as a contour line on a map joins points of equal elevation: closely
spaced lines mean the count changes quickly, and the innermost closed loop is the peak. It is the same histogram as the cell above drawn a third way, with the axes exchanged — cholesterol
runs horizontally here and maximum heart rate vertically — so the innermost loop sits at the same
pair of values where that histogram is densest. The lines are drawn at the left edge of each bin
rather than its centre, so everything is displaced by half a bin. For the same information this is usually the easiest of the three to read, and it is the
standard picture for a function of two variables being minimized, which is why contour plots recur
throughout machine learning.

Part 1 reduces to a short procedure, and it is the procedure to run on any new dataset before
anything else: look at normalized distributions of the random variables for the different
populations, then at scatter plots of the features in pairs, then at the pair plot that combines
them. Every word in that sentence has now been defined. The reason to insist on it is that the
alternative — starting work on the data before looking at it — regularly ends in time spent on data
that was broken, or on a question the data could never have answered.

# Part 2 — Does It Look Normal?

Look back at the histograms in Part 1. Some are roughly bell-shaped. Many are not:
`chol` has a long right tail, `thalach` leans left, `oldpeak` piles up against zero.

The reason we *expect* bell shapes is the **Central Limit Theorem (CLT)**: add up enough
independent random pieces and the sum is approximately normal, no matter what the pieces
look like. That is why the normal distribution shows up for measurement noise, heights,
sums of dice and detector readings. But "enough independent pieces" hides four separate
assumptions:

| Assumption | What it means | What breaks it |
|---|---|---|
| **Identically distributed** | every piece comes from the same distribution | mixing populations |
| **Independent** | knowing one piece tells you nothing about the next | correlated samples |
| **Finite variance** | no piece can be wildly, unboundedly large | heavy tails |
| **Additive** | the pieces add | pieces that multiply |

Real data guarantees none of these. This part is a *diagnosis*, not a list to memorize:
you histogram something, it does not look normal — which assumption broke, and which plot
would tell you? Each section follows the same pattern:

1. generate data with a **known** cause,
2. histogram it, to see how the problem presents,
3. make the plot that reveals the cause, as if you didn't know it,
4. name the assumption that broke.

Each failure comes with a physics example and a data-science example of exactly the same
mathematics — the failure is a property of the math, not of the field.

In [ ]:
from scipy import stats

np.random.seed(3402)

## What success looks like

First, the CLT working. Each value below is the sum of 12 uniform random numbers. A single
uniform is flat — nothing like a bell — but the sum of twelve is very close to normal.

**The math.** Let $X_1, X_2, \ldots, X_n$ be independent draws from one distribution with
mean $\mu$ and variance $\sigma^2$. Their sum $S_n = \sum_{i=1}^{n} X_i$ and their mean
$\bar{X}_n = S_n / n$ have

$$
E[S_n] = n\mu, \qquad \mathrm{Var}(S_n) = n\sigma^2, \qquad
E[\bar{X}_n] = \mu, \qquad \mathrm{Var}(\bar{X}_n) = \frac{\sigma^2}{n}.
$$

Those four facts are just bookkeeping. The CLT is the surprising part: the *shape* becomes
normal,

$$
\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \;\longrightarrow\; \mathcal{N}(0, 1)
\quad \text{as } n \to \infty,
$$

whatever the shape of the distribution of a single $X_i$. Each of the four assumptions
appears in that derivation: *one* $\mu$ and $\sigma$ (identically distributed), the
variance of the sum being the sum of the variances (independent), $\sigma^2 < \infty$
(finite variance), and $S_n$ being a sum (additive).

A uniform on $[0, 1]$ has $\mu = \tfrac{1}{2}$ and $\sigma^2 = \tfrac{1}{12}$. So the sum
of 12 of them has mean $6$ and variance exactly $1$ — that is why twelve is the classic
choice.

In [ ]:
pieces = np.random.uniform(0, 1, size=(5000, 12))
sums = pieces.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(pieces[:, 0], bins=40, density=True)
axes[0].set_title("one piece: uniform")
axes[1].hist(sums, bins=40, density=True)
axes[1].set_title("sum of 12 pieces")
plt.show()

**Code walkthrough.**

* `np.random.uniform(0, 1, size=(5000, 12))` makes a 5000 × 12 array: each **row** is one
  experiment, each of its 12 **columns** is one piece.
* `.sum(axis=1)` adds across the columns of each row, giving 5000 sums. (`axis=0` would
  instead add down each column.)
* `pieces[:, 0]` is the first column — 5000 single pieces, to show what one piece looks like.
* `plt.subplots(1, 2)` returns a figure and an array of two `axes`. Drawing on
  `axes[0]` and `axes[1]` instead of `plt` is how you put several plots side by side; we'll
  use this pattern for the rest of the lecture.
* `density=True` scales each histogram to unit area, so it can be compared with a
  probability density.

Check the prediction: the sum should have mean 6 and standard deviation 1.

In [ ]:
print("mean:", np.mean(sums), "  std:", np.std(sums))

## Tool 1: overlay the normal you'd expect

"Looks normal" is easier to judge against a reference. Draw the normal curve with the
sample's own mean and standard deviation on top of the density histogram. Where the
histogram and the curve disagree is where to look.

**The math.** The normal density with mean $\mu$ and standard deviation $\sigma$ is

$$
f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left( -\frac{(x - \mu)^2}{2\sigma^2} \right).
$$

We don't know $\mu$ and $\sigma$, so we plug in the sample estimates
$\hat\mu = \frac{1}{n}\sum_i x_i$ and $\hat\sigma = \sqrt{\frac{1}{n}\sum_i (x_i - \hat\mu)^2}$.

In [ ]:
def hist_with_normal(x, bins=50, ax=None, label=None):
    if ax is None:
        ax = plt.gca()
    x = np.asarray(x)
    _, edges, _ = ax.hist(x, bins=bins, density=True, alpha=0.6, label=label)
    grid = np.linspace(edges[0], edges[-1], 300)
    ax.plot(grid, stats.norm.pdf(grid, np.mean(x), np.std(x)), "k--", label="normal, same mean & std")
    ax.legend(fontsize=8)


hist_with_normal(sums)

**Code walkthrough.**

* `ax=None` with `if ax is None: ax = plt.gca()` lets the function draw either on the
  current plot or on one panel of a subplot grid that we pass in. Every helper in this part
  follows that pattern, which is what makes the diagnostic panel at the end possible.
* `ax.hist` returns three things: counts, bin **edges**, and the drawn patches. We keep the
  edges so the curve covers exactly the histogram's range.
* `density=True` is essential here: a density histogram has area 1, as $f(x)$ does. A
  histogram of raw counts would sit $n \times$ (bin width) above the curve.
* `stats.norm.pdf(grid, mean, std)` evaluates $f(x)$ at the 300 points of `grid`.

## Tool 2: the quantile–quantile (Q–Q) plot

Histograms depend on binning and hide the tails, which is exactly where most departures
from normality live. A **Q–Q plot** has neither problem:

* sort the data — the $i$-th smallest value is the sample's $\frac{i - 0.5}{n}$ quantile;
* compute where that same quantile falls for a standard normal (`stats.norm.ppf`);
* scatter one against the other.

**The math.** Sort the data: $x_{(1)} \le x_{(2)} \le \cdots \le x_{(n)}$. The $i$-th
smallest value estimates the quantile at probability $p_i = \frac{i - 0.5}{n}$ (the
$-0.5$ keeps $p$ strictly between 0 and 1). If $\Phi$ is the standard normal CDF, the
matching normal quantile is $z_i = \Phi^{-1}(p_i)$. If the data really are
$\mathcal{N}(\mu, \sigma^2)$, then

$$
x_{(i)} \approx \mu + \sigma\, z_i ,
$$

a straight line in $(z_i, x_{(i)})$ with slope $\sigma$ and intercept $\mu$. Any departure
from a line is a departure from normality, and *where* it departs tells you which tail.

For the reference line we want a $\mu$ and $\sigma$ that a few extreme points can't drag
around, so we use the median for $\mu$ and get $\sigma$ from the interquartile range. For a
normal distribution the quartiles sit at $\mu \pm 0.674\,\sigma$, so

$$
\hat\sigma = \frac{Q_3 - Q_1}{\Phi^{-1}(0.75) - \Phi^{-1}(0.25)} = \frac{\mathrm{IQR}}{1.349}.
$$

In [ ]:
def qq_plot(x, ax=None, label=None):
    if ax is None:
        ax = plt.gca()
    x = np.sort(np.asarray(x))
    n = len(x)
    theoretical = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
    ax.scatter(theoretical, x, s=4, label=label)

    q25, q50, q75 = np.percentile(x, [25, 50, 75])
    slope = (q75 - q25) / (stats.norm.ppf(0.75) - stats.norm.ppf(0.25))
    ax.plot(theoretical, q50 + slope * theoretical, "k--", lw=1)
    ax.set_xlabel("normal quantile")
    ax.set_ylabel("data quantile")


qq_plot(sums)

**Code walkthrough.**

* `np.arange(1, n + 1)` is $i = 1, \ldots, n$, so `(np.arange(1, n + 1) - 0.5) / n` is the
  whole array of $p_i$ at once — no loop.
* `stats.norm.ppf` is $\Phi^{-1}$. ("ppf" is scipy's name for the inverse CDF, the *percent
  point function*.)
* `np.percentile(x, [25, 50, 75])` returns $Q_1$, the median and $Q_3$ in one call.
* `slope` is $\hat\sigma = \mathrm{IQR}/1.349$, and the dashed line is
  $\text{median} + \hat\sigma z$.

How to read a Q–Q plot — each failure in this part has its own shape:

| Shape | Meaning |
|---|---|
| straight line | consistent with normal |
| bends **up** at the right end only | long right tail (right skew) — e.g. products |
| bends **down** at the left end only | long left tail (left skew) |
| peels away at **both** ends, steeply | heavy tails |
| flat step in the middle, or an S | two populations (a mixture) |
| horizontal runs of points | discrete or rounded values |

### Real data first

Back to `chol` from Part 1:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hist_with_normal(df["chol"], bins=40, ax=axes[0])
qq_plot(df["chol"], ax=axes[1])
plt.show()

The bulk tracks the line, but the right end bends up and away — more high cholesterol
values than a normal distribution allows, with one patient near 560 mg/dl. We'll come back
to `chol` and explain this shape in Section 4. First, the four failures one at a time.

## 1. Mixtures of populations

Pool two different populations into one sample and the first assumption breaks: the
pieces no longer come from the *same* distribution. The result is two humps, or one hump
with a shoulder, or just a distribution that is fatter or more lopsided than it should be.
And the overall mean describes no actual member of either population.

**Data science.** Page load time measured across all visitors is two populations pooled
together: mobile and desktop users have entirely different distributions.

**The math.** A mixture of two populations, a fraction $w_1$ from density $f_1$ and
$w_2 = 1 - w_1$ from $f_2$, has density

$$
f(x) = w_1 f_1(x) + w_2 f_2(x).
$$

Its mean and variance are

$$
\mu = w_1 \mu_1 + w_2 \mu_2, \qquad
\sigma^2 = \underbrace{w_1 \sigma_1^2 + w_2 \sigma_2^2}_{\text{spread within groups}}
\;+\; \underbrace{w_1 w_2 (\mu_1 - \mu_2)^2}_{\text{spread between groups}} .
$$

The second term is why a mixture looks "too wide": the separation between the groups adds
to the variance even when each group is narrow.

In [ ]:
mobile = np.random.normal(2.5, 0.4, 5000)
desktop = np.random.normal(0.8, 0.2, 5000)
load_time = np.concatenate([mobile, desktop])

hist_with_normal(load_time, bins=60)
plt.xlabel("page load time (s)")

**Code walkthrough.** `np.concatenate` stacks the two arrays end to end into one array of
10,000 values. That one line *is* the pooling: from here on nothing records which value
came from which device. The dashed normal uses the pooled mean and standard deviation, and
fits neither group.

**Physics.** An invariant-mass spectrum from a particle detector is signal sitting on
background: a narrow peak from the particle you are looking for, on a falling
distribution from everything else that left a similar signature.

In [ ]:
signal = np.random.normal(125, 2, 2000)
background = 80 + np.random.exponential(30, 8000)
mass = np.concatenate([signal, background])

_ = plt.hist(mass, bins=80)
plt.xlabel("invariant mass (GeV)")

**Code walkthrough.** `np.random.exponential(30, 8000)` draws a falling distribution with
mean 30, starting at 0. Adding 80 shifts it to start at 80 GeV, like a background that
turns on at a trigger threshold. The signal is 2,000 events against 8,000 background,
$w_1 = 0.2$.

Same failure, different sizes: the load-time humps are equal and far apart, while the
signal here is a small bump on a big slope. Now look at what the pooled mean means:

In [ ]:
print("pooled :", np.mean(load_time))
print("mobile :", np.mean(mobile))
print("desktop:", np.mean(desktop))

No mobile user and no desktop user has a typical load time of about 1.65 s. That number
belongs to a population that doesn't exist. It is exactly $w_1\mu_1 + w_2\mu_2 =
0.5 \times 2.5 + 0.5 \times 0.8 = 1.65$. Check the variance formula too:

In [ ]:
w1, w2 = 0.5, 0.5
predicted_var = w1 * 0.4**2 + w2 * 0.2**2 + w1 * w2 * (2.5 - 0.8)**2
print("predicted variance:", predicted_var)
print("measured variance :", np.var(load_time))

The within-group part is only $0.1$; the between-group part, $0.25 \times 1.7^2 = 0.72$,
is seven times larger.

### How far apart do two populations have to be before you see two humps?

Less often than you'd hope. Below, two equal-sized normal populations with the same width
$\sigma$, with their means moved apart by $\Delta\mu$:

In [ ]:
sigma = 1.0
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, separation in zip(axes, [0.5, 1.0, 2.0, 3.0]):
    group_1 = np.random.normal(0, sigma, 5000)
    group_2 = np.random.normal(separation * sigma, sigma, 5000)
    pooled = np.concatenate([group_1, group_2])
    _, bins, _ = ax.hist(pooled, bins=50, color="gray", alpha=0.4, label="pooled")
    ax.hist(group_1, bins=bins, histtype="step", label="group 1")
    ax.hist(group_2, bins=bins, histtype="step", label="group 2")
    ax.set_title("$\\Delta\\mu$ = " + str(separation) + "$\\sigma$")

axes[0].legend(fontsize=8)
plt.show()

**Code walkthrough.**

* `zip(axes, [0.5, 1.0, 2.0, 3.0])` pairs each of the four panels with one separation, so a
  single loop fills the whole row.
* The pooled histogram is drawn first and its bin edges `bins` are reused for both groups,
  so all three histograms share bins.
* `histtype="step"` draws outlines only, so the groups stay visible on top of the filled
  pooled histogram.
* The title is built from `separation`, the number the code actually used, so the title
  and the data can't disagree.

Below $2\sigma$ the pooled histogram has **one** peak; at $2\sigma$ it is just flat-topped.

**The math.** For an equal mixture of two normals with the same $\sigma$, put the origin
halfway between the means. The density is
$f(x) \propto e^{-(x - d)^2 / 2\sigma^2} + e^{-(x + d)^2 / 2\sigma^2}$ with
$d = \Delta\mu / 2$. At $x = 0$ its second derivative is proportional to $d^2 - \sigma^2$.
So the midpoint is a peak (one hump) while $d < \sigma$, and becomes a dip (two humps)
only once $d > \sigma$, that is, once $\Delta\mu > 2\sigma$.
Until then the mixture only shows up as a distribution that is too wide, too flat, or — if
the groups are unequal in size — skewed.
So you can't rely on spotting two humps.

**Diagnostic: split by every covariate you have.** You rarely get a variable with nothing
else attached: you get device type, browser, time of day, a diagnosis. Split on the right
one and each part looks like one clean distribution. We already built the tool for this —
`compare_distributions` from Part 1.

**Real data.** Pooled, maximum heart rate `thalach` leans to the left. Split by the
disease label:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hist_with_normal(df["thalach"], bins=30, ax=axes[0])
qq_plot(df["thalach"], ax=axes[1])
plt.show()

compare_distributions(df, "thalach",
                      {"no disease": df["target"] == 0, "disease": df["target"] == 1},
                      bins=25, alpha=0.5, density=1)
plt.xlabel("thalach")

The two groups are centered about 20 beats per minute apart. Neither hump is visible in
the pooled histogram — less than one standard deviation of separation, exactly as in the
$\Delta\mu = 0.5\sigma$–$1\sigma$ panels above — but the split shows it plainly. Each group
on its own is still a bit left-skewed (heart rate has a ceiling that depends on age), so
the mixture isn't the whole story; it is one ingredient you would never find by staring
at the pooled histogram.

**Assumption broken: identically distributed.**

## 2. Correlated samples

Now every draw *is* from the same distribution, but the draws are not independent —
knowing one tells you something about the next. The consequence is easy to miss:
**the variance of the mean stops falling like $1/n$.** A thousand rows of correlated data
can carry the statistical weight of a few dozen independent ones.

**Physics.** Pile-up: several collisions in the same readout window share an underlying
vertex. Or a calibration drifts slowly through a run and nudges every nearby measurement
the same way.

**Data science.** The same customer measured again and again, or any time series —
today's value is close to yesterday's, not a fresh draw.

**The math.** The simplest correlated sequence is the *autoregressive* process AR(1):

$$
x_i = \phi\, x_{i-1} + \varepsilon_i, \qquad
\varepsilon_i \sim \mathcal{N}\!\left(0,\; 1 - \phi^2\right), \qquad 0 \le \phi < 1 .
$$

Each value keeps a fraction $\phi$ of the previous one and adds fresh noise. Why
$1 - \phi^2$? Taking the variance of both sides,
$\mathrm{Var}(x) = \phi^2\,\mathrm{Var}(x) + (1 - \phi^2)$, which is solved by
$\mathrm{Var}(x) = 1$. So every $x_i$ is $\mathcal{N}(0, 1)$, exactly like the independent
sequence; only the ordering differs. Values $k$ steps apart have correlation

$$
\rho_k = \mathrm{corr}(x_i, x_{i+k}) = \phi^k .
$$

Build one sequence with no correlation and one with $\phi = 0.95$, matched so that the
histogram of values (which ignores order) is the same:

In [ ]:
def ar1(n, phi):
    # each value is phi times the previous one, plus fresh noise
    sigma = (1 - phi ** 2) ** 0.5
    x = np.zeros(n)
    x[0] = np.random.normal(0, 1)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + np.random.normal(0, sigma)
    return x


independent = np.random.normal(0, 1, 3000)
correlated = ar1(3000, 0.95)

_, bins, _ = plt.hist(independent, bins=40, alpha=0.5, density=1, label="independent")
_ = plt.hist(correlated, bins=bins, alpha=0.5, density=1, label="correlated")
_ = plt.legend()

**Code walkthrough.** `ar1` is the equation above written as a loop: start from one
$\mathcal{N}(0, 1)$ draw, then build each `x[i]` from `x[i - 1]`. A loop is needed here
because each value depends on the previous one, so numpy can't compute them all at once.
`sigma` is the standard deviation of the fresh noise, $\sqrt{1 - \phi^2}$.

Same mean, same spread, same shape. The histogram can't tell them apart, because the
difference is entirely in the **order** of the values, which a histogram throws away.

**Diagnostic 1: plot the values in order.** This is just a 1-D plot of the kind from
Part 1, with the row number on the x-axis:

In [ ]:
_ = plt.plot(independent[:300], label="independent")
_ = plt.plot(correlated[:300], label="correlated")
_ = plt.xlabel("row number")
_ = plt.legend()

The independent sequence jitters; the correlated one wanders in slow swings.

**Diagnostic 2: the autocorrelation plot.** Shift the sequence against itself by a *lag*
and compute the correlation; do that for a range of lags.

**The math.** The sample autocorrelation at lag $k$ is

$$
\hat\rho_k = \frac{\sum_{i=1}^{n-k} (x_i - \bar{x})(x_{i+k} - \bar{x})}
                  {\sum_{i=1}^{n} (x_i - \bar{x})^2} .
$$

At $k = 0$ it is exactly 1. For independent data it should be near 0 at every other lag.
For AR(1) it should follow $\phi^k$, drawn as a dashed line below.

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x) - np.mean(x)
    denominator = np.sum(x * x)
    result = list()
    for lag in range(max_lag + 1):
        result.append(np.sum(x[:len(x) - lag] * x[lag:]) / denominator)
    return result


_ = plt.plot(autocorrelation(independent, 60), marker=".", label="independent")
_ = plt.plot(autocorrelation(correlated, 60), marker=".", label="correlated")
_ = plt.plot(0.95 ** np.arange(61), "k--", label="prediction: 0.95 ** k")
_ = plt.axhline(0, color="k", lw=0.5)
_ = plt.xlabel("lag")
_ = plt.ylabel("autocorrelation")
_ = plt.legend()

**Code walkthrough.**

* `x - np.mean(x)` centers the data once, so the formula's $(x_i - \bar{x})$ terms are just
  `x`.
* For a given `lag`, `x[:len(x) - lag]` is $x_1, \ldots, x_{n-k}$ and `x[lag:]` is
  $x_{1+k}, \ldots, x_n$. Those two slices have the same length, and multiplying them
  element by element pairs every value with the one $k$ steps later. `np.sum` of the
  product is the numerator.
* `0.95 ** np.arange(61)` is $\phi^k$ for $k = 0, \ldots, 60$, computed for all $k$ at once.

The independent series drops to about zero the moment the lag is not zero. The correlated
series decays slowly — that slow decay is the signature. It follows the $\phi^k$ prediction
in shape but not point for point. That isn't a bug: $\hat\rho_k$ is itself estimated from
one sequence whose effective sample size is only about $3000/39 \approx 77$, and
neighboring lags share almost the same data, so their errors move together. Rerun the cell
and the curve will land somewhere else around the prediction.

**Diagnostic 3: check the $1/n$ scaling directly.** Take the mean of $n$ points, repeat
many times, and see how the variance of *that* mean falls as $n$ grows. On log–log axes
$1/n$ is a straight line of slope $-1$.

**The math.** The variance of a mean is the sum of every pairwise covariance:

$$
\mathrm{Var}(\bar{x}) = \frac{1}{n^2} \sum_{i=1}^{n} \sum_{j=1}^{n} \mathrm{Cov}(x_i, x_j)
= \frac{\sigma^2}{n} \left[ 1 + 2 \sum_{k=1}^{n-1} \left(1 - \frac{k}{n}\right) \rho_k \right].
$$

For independent data every $\rho_k = 0$ and only the familiar $\sigma^2/n$ survives. For
AR(1), $\rho_k = \phi^k$, and for $n$ much larger than the correlation length the bracket
tends to $\frac{1 + \phi}{1 - \phi}$:

$$
\mathrm{Var}(\bar{x}) \approx \frac{\sigma^2}{n} \cdot \frac{1 + \phi}{1 - \phi}
= \frac{\sigma^2}{n_\text{eff}}, \qquad
n_\text{eff} = n \, \frac{1 - \phi}{1 + \phi} .
$$

For $\phi = 0.95$ that factor is $39$: every 39 rows carry the information of one
independent row.

In [ ]:
def variance_of_the_mean(generate, n, trials):
    means = list()
    for _ in range(trials):
        means.append(np.mean(generate(n)))
    return np.var(means)


ns = [10, 30, 100, 300, 1000]
var_independent = [variance_of_the_mean(lambda n: np.random.normal(0, 1, n), n, 300) for n in ns]
var_correlated = [variance_of_the_mean(lambda n: ar1(n, 0.95), n, 300) for n in ns]

_ = plt.plot(ns, var_independent, marker="o", label="independent")
_ = plt.plot(ns, var_correlated, marker="o", label="correlated")
_ = plt.plot(ns, 1 / np.array(ns), "k--", label="1/n")


def ar1_variance_of_the_mean(n, phi):
    k = np.arange(1, n)
    return (1 + 2 * np.sum((1 - k / n) * phi ** k)) / n


_ = plt.plot(ns, [ar1_variance_of_the_mean(n, 0.95) for n in ns], "k:", label="prediction for AR(1)")
plt.xscale("log")
plt.yscale("log")
_ = plt.xlabel("n")
_ = plt.ylabel("variance of the mean")
_ = plt.legend()

**Code walkthrough.**

* `variance_of_the_mean` takes a *function* `generate` as its first argument, so the same
  code works for both kinds of data. `lambda n: ar1(n, 0.95)` is a one-line unnamed
  function that turns `ar1` into something that only needs `n`.
* It repeats the experiment 300 times, keeps the 300 means, and returns the variance of
  those means — the left-hand side of the equation, measured.
* `ar1_variance_of_the_mean` is the right-hand side, with $\sigma^2 = 1$ and
  $\rho_k = \phi^k$. `np.arange(1, n)` is $k = 1, \ldots, n-1$, so the sum over $k$ is a
  single `np.sum`.
* `plt.xscale("log")` and `plt.yscale("log")` make power laws straight lines.

The independent series sits on the $1/n$ line. The correlated series sits far above it
at every $n$ and falls much more slowly. Push $n$ well past the correlation length and
$1/n$ scaling does come back, but with a much smaller *effective* sample size — for this
sequence about $n \frac{1-\phi}{1+\phi} = n/39$. The row count is what you can see; the
effective sample size is what sets your uncertainty.

**Assumption broken: independence.**

## 3. Heavy tails

This is the assumption everyone forgets: each piece needs **finite variance**. The
textbook case is the **Cauchy distribution**, whose variance is infinite. The consequence
is not slow convergence — there is no convergence. The mean of $n$ Cauchy draws is itself
Cauchy, with exactly the same width as a single draw. Averaging buys nothing.

**Data science.** Network latency, insurance claims, file-transfer sizes — anything with
power-law tails, where the sample mean keeps wandering as data comes in.

**The math.** The Cauchy density with center $x_0$ and width $\gamma$ is

$$
f(x) = \frac{1}{\pi\gamma} \cdot \frac{1}{1 + \left( \frac{x - x_0}{\gamma} \right)^2} .
$$

It looks like a bell, but its tails fall only like $1/x^2$, whereas a normal's fall like
$e^{-x^2/2}$. That difference is fatal for averaging. The mean requires
$\int |x| f(x)\, dx$, and for large $|x|$ the integrand behaves like $|x| \cdot 1/x^2 = 1/|x|$,
whose integral grows like $\log |x|$ without limit. So the Cauchy distribution has **no
mean**, and therefore no variance either.

What happens to an average of Cauchy draws? The cleanest tool is the characteristic
function, $\varphi(t) = E[e^{itX}]$, which for Cauchy is $e^{i x_0 t - \gamma |t|}$. For a
mean of $n$ independent draws,

$$
\varphi_{\bar{X}}(t) = \left[ \varphi(t/n) \right]^n
= \left( e^{i x_0 t/n - \gamma |t|/n} \right)^n = e^{i x_0 t - \gamma |t|} ,
$$

which is exactly the characteristic function of a single draw. **The mean of $n$ Cauchy
draws has the same distribution as one draw**, for every $n$.

**First clue: the histogram looks broken.**

In [ ]:
gaussian_sample = np.random.normal(0, 1, 4000)
cauchy_sample = np.random.standard_cauchy(4000)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(gaussian_sample, bins=50)
axes[0].set_title("Gaussian")
axes[1].hist(cauchy_sample, bins=50)
axes[1].set_title("Cauchy")
plt.show()

print("Cauchy min and max:", cauchy_sample.min(), cauchy_sample.max())

**Code walkthrough.** `np.random.standard_cauchy` draws with $x_0 = 0$ and $\gamma = 1$.
`plt.hist` with `bins=50` divides the full range, minimum to maximum, into 50 equal bins.
When that range is thousands of units wide, each bin is wider than the whole bulk of the
data, so almost everything lands in one bin.

The Cauchy histogram is one spike: a handful of draws in the hundreds or thousands set the
axis range and squash everything else into a bin or two. When a few points dictate the
whole axis, suspect heavy tails.

**Diagnostic: the CLT, directly.** Histogram the mean of $n$ draws for growing $n$. For
the Gaussian the distribution of the mean narrows like $1/\sqrt{n}$. For Cauchy it
doesn't narrow at all:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bins = np.linspace(-5, 5, 61)
for n in [1, 10, 100]:
    gauss_means = np.random.normal(0, 1, size=(2000, n)).mean(axis=1)
    cauchy_means = np.random.standard_cauchy(size=(2000, n)).mean(axis=1)
    axes[0].hist(gauss_means, bins=bins, histtype="step", density=True, label="n = " + str(n))
    axes[1].hist(cauchy_means, bins=bins, histtype="step", density=True, label="n = " + str(n))
axes[0].set_title("mean of n Gaussian draws")
axes[1].set_title("mean of n Cauchy draws")
axes[0].legend()
axes[1].legend()
plt.show()

**Code walkthrough.** `size=(2000, n)` makes 2,000 rows of `n` draws, and `.mean(axis=1)`
averages each row: 2,000 repeated experiments, each averaging `n` values. The bin edges
are fixed with `np.linspace(-5, 5, 61)` so all three histograms share them. With fixed
edges, a narrowing distribution looks narrower, and a Cauchy one visibly stays put.

The left panel is $\sigma/\sqrt{n}$ in action: the width shrinks by $\sqrt{10} \approx 3$
each time $n$ grows tenfold. The right panel is the characteristic-function result: three
identical histograms.

In real data you only have one sample, so you can't repeat the experiment. The practical
version is the **running mean**: the mean of the first $k$ points, plotted against $k$,

$$
\bar{x}_k = \frac{1}{k} \sum_{i=1}^{k} x_i , \qquad k = 1, 2, \ldots, n .
$$

The law of large numbers says $\bar{x}_k$ settles on the true mean — *if one exists* — and
the CLT says the wobble around it shrinks like $\sigma/\sqrt{k}$ — *if $\sigma$ is finite*.

In [ ]:
def running_mean(x):
    x = np.asarray(x)
    return np.cumsum(x) / np.arange(1, len(x) + 1)


_ = plt.plot(running_mean(gaussian_sample), label="Gaussian")
_ = plt.plot(running_mean(cauchy_sample), label="Cauchy")
_ = plt.xlabel("number of points")
_ = plt.ylabel("running mean")
_ = plt.legend()

**Code walkthrough.** `np.cumsum(x)` is the array of partial sums
$x_1,\; x_1 + x_2,\; x_1 + x_2 + x_3, \ldots$. Dividing element by element by
`np.arange(1, len(x) + 1)` $= 1, 2, 3, \ldots$ gives every running mean in one line, with no
loop.

The Gaussian running mean settles within a few hundred points. The Cauchy one lurches
every time a big draw arrives, and big draws never stop arriving.

And the Q–Q plot: heavy tails peel away from the line at **both** ends.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
qq_plot(gaussian_sample, ax=axes[0])
axes[0].set_title("Gaussian")
qq_plot(cauchy_sample, ax=axes[1])
axes[1].set_title("Cauchy (y-axis clipped)")
axes[1].set_ylim(-30, 30)
plt.show()

The y-axis on the right is clipped on purpose — remove the `set_ylim` and the line collapses
into a dot. Needing to clip is itself the diagnosis.

### The same failure can hide in your instrument

The parent distribution is not the only place infinite variance can enter. Averaging
**repeated measurements of one fixed quantity** is the same sum, and it fails the same way
if the *measurement noise* — the resolution or response function — is heavy-tailed.
Physics examples: Landau straggling in a thin absorber, or any ratio whose denominator can
land near zero.

The trap: **this has nothing to do with whether the resolution is good or bad.** A wide
Gaussian instrument still averages down; it just needs more measurements. Only an
infinite-variance noise distribution breaks averaging, however precise it looks.

**The math.** Each measurement is the true value plus noise, $y_i = \theta + \epsilon_i$, so

$$
\bar{y}_n = \theta + \bar{\epsilon}_n .
$$

The estimate is only as good as the average of the noise. For Gaussian noise of width $s$,
$\bar\epsilon_n$ has width $s/\sqrt{n}$: a wide instrument (large $s$) just needs a larger
$n$. For Cauchy noise of width $\gamma$, $\bar\epsilon_n$ is Cauchy with width $\gamma$ —
the same as a single measurement, for every $n$.

In [ ]:
true_value = 10.0
n = 5000
instruments = {"narrow Gaussian noise": lambda: np.random.normal(0, 0.2, n),
               "wide Gaussian noise": lambda: np.random.normal(0, 4.0, n),
               "narrow Cauchy noise": lambda: np.random.standard_cauchy(n) * 0.2}

# five independent runs of each instrument, measuring the same true value
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (label, noise) in zip(axes, instruments.items()):
    for run in range(5):
        ax.plot(running_mean(true_value + noise()), lw=1)
    ax.axhline(true_value, color="k", lw=0.5)
    ax.set_title(label)
    ax.set_xlabel("number of measurements")
axes[0].set_ylim(true_value - 1.5, true_value + 1.5)
plt.show()

**Code walkthrough.**

* `instruments` is a dictionary from a name to a function that produces `n` noise values.
  Storing functions (the `lambda`s) instead of arrays means each call draws **fresh** noise,
  which is what lets us repeat each instrument five times.
* `zip(axes, instruments.items())` gives one panel per instrument; the inner loop draws five
  independent runs on that panel.
* `sharey=True` gives all three panels the same y-axis, so the instruments are compared on
  equal footing; setting the limit on `axes[0]` sets it for all three.

Five runs of each instrument. The wide Gaussian runs start noisier but all converge on the
true value. The "narrow" Cauchy runs — the most precise-looking instrument, judged by the
width of its central peak — keep jumping and end up disagreeing with each other: more
measurements don't make them agree.

**Assumption broken: finite variance.**

## 4. Products instead of sums

The last assumption seems too obvious to state: the pieces **add**. Plenty of real
quantities **multiply** instead, and the CLT says nothing directly about a product.

But a product is a sum in disguise.

**The math.** If $G = \prod_{i=1}^{n} g_i$ with every $g_i > 0$, then

$$
\log G = \sum_{i=1}^{n} \log g_i ,
$$

a sum of independent pieces, so the CLT applies to the log:
$\log G \approx \mathcal{N}(m, s^2)$ with $m = n\, E[\log g]$ and $s^2 = n\, \mathrm{Var}(\log g)$.
Then $G$ itself is **log-normal**, with density

$$
f(G) = \frac{1}{G\, s \sqrt{2\pi}} \exp\!\left( -\frac{(\ln G - m)^2}{2 s^2} \right), \qquad G > 0 .
$$

It is skewed to the right: the median is $e^{m}$ but the mean is $e^{m + s^2/2}$, pulled up
by the long right tail. Multiplying by a factor above 1 several times in a row compounds
far more than the same factors would add.

**Physics.** A photomultiplier gain chain: each dynode multiplies the signal by a random
factor, and the total gain is the product of every stage.

**Data science.** Income, session length, file size — anything built from compounding
multiplicative factors.

In [ ]:
stage_gains = np.random.uniform(0.7, 1.3, size=(4000, 15))
total_gain = np.prod(stage_gains, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hist_with_normal(total_gain, ax=axes[0])
qq_plot(total_gain, ax=axes[1])
plt.show()

**Code walkthrough.** `np.random.uniform(0.7, 1.3, size=(4000, 15))` gives 4,000 trials
(rows) of 15 stage gains (columns), each between 0.7 and 1.3. `np.prod(..., axis=1)`
multiplies across each row, the same way `.sum(axis=1)` added across rows in the very first
example of this part. That one change, `prod` instead of `sum`, is the entire difference.

Skewed right, and the Q–Q plot bends up at the right end only. A few trials that drew high
gains at every stage compound into totals far above the bulk. A sum of the same fifteen
numbers would never do that.

**Diagnostic: histogram the log.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hist_with_normal(np.log(total_gain), ax=axes[0])
qq_plot(np.log(total_gain), ax=axes[1])
plt.show()

The log is the sum the CLT was talking about, and it is normal.

**Real data.** Back to `chol`, whose Q–Q plot bent up on the right at the start of this
part. To put a number on the asymmetry, use the **skewness**,

$$
\gamma_1 = \frac{E\left[ (x - \mu)^3 \right]}{\sigma^3} .
$$

Cubing keeps the sign of each deviation, so a long right tail makes $\gamma_1 > 0$, a long
left tail makes it negative, and any symmetric distribution, the normal included, has
$\gamma_1 = 0$. `stats.skew` computes the sample version.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
hist_with_normal(np.log(df["chol"]), bins=40, ax=axes[0])
axes[0].set_xlabel("log(chol)")
qq_plot(np.log(df["chol"]), ax=axes[1])
plt.show()

print("skewness of chol     :", stats.skew(df["chol"]))
print("skewness of log(chol):", stats.skew(np.log(df["chol"])))

The log removes almost all the skew (one patient near 560 mg/dl is still an outlier).
Careful with the conclusion: "the log looks normal" is *consistent* with a multiplicative
mechanism, not proof of one. But it is a strong hint about which scale to analyze the
variable on — and it is why log axes are so common in plots of incomes, sizes and rates.

**Assumption broken: addition.**

## Not every non-normal histogram is a CLT failure

The CLT only makes a promise about quantities that *are* sums of many small, independent
contributions. Plenty of variables never were. `oldpeak` (ST depression) is bounded below
by zero, and a large fraction of patients sit exactly at zero:

In [ ]:
compare_distributions(df, "oldpeak",
                      {"no disease": df["target"] == 0, "disease": df["target"] == 1},
                      bins=25, alpha=0.5, density=1)
plt.xlabel("oldpeak")
print("fraction exactly zero:", np.mean(df["oldpeak"] == 0))

No logarithm or covariate split will make that normal, and nothing has gone wrong:
**bounded, discrete, and zero-inflated** variables simply aren't sums. The same goes for
the categorical columns (`cp`, `thal`, `ca`, …), which is why Part 1 summarized those with
tables instead. Before reaching for a diagnosis, ask whether there was any reason to expect
a bell curve in the first place.

## Putting it together: one diagnostic panel

Each failure has a plot that exposes it. Since you don't know in advance which one you're
facing, make all of them at once — one function, one grid of subplots:

In [ ]:
def diagnose(x, group=None, name=""):
    x = np.asarray(x)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(name)

    hist_with_normal(x, ax=axes[0, 0])
    axes[0, 0].set_title("histogram vs normal")

    qq_plot(x, ax=axes[0, 1])
    axes[0, 1].set_title("Q-Q plot")

    if np.all(x > 0):
        hist_with_normal(np.log(x), ax=axes[0, 2])
        axes[0, 2].set_title("histogram of log(x)")
    else:
        axes[0, 2].set_title("log: not possible (values <= 0)")

    axes[1, 0].plot(running_mean(x))
    axes[1, 0].set_title("running mean")

    axes[1, 1].plot(autocorrelation(x, 50), marker=".")
    axes[1, 1].axhline(0, color="k", lw=0.5)
    axes[1, 1].set_title("autocorrelation")

    if group is not None:
        bins = np.histogram_bin_edges(x, bins=40)
        axes[1, 2].hist(x[group == 0], bins=bins, alpha=0.5, density=True, label="group 0")
        axes[1, 2].hist(x[group == 1], bins=bins, alpha=0.5, density=True, label="group 1")
        axes[1, 2].legend()
    axes[1, 2].set_title("split by covariate")

    plt.tight_layout()
    plt.show()


diagnose(df["thalach"].values, df["target"].values, "thalach, split by target")

**Code walkthrough.**

* `plt.subplots(2, 3)` returns a 2 × 3 array of axes; `axes[row, column]` picks a panel.
  Every helper we wrote takes an `ax=` argument, which is what lets `diagnose` place each
  plot in its own panel.
* The top row tests **shape**: overlay, Q–Q, and log. `np.all(x > 0)` guards the log
  panel, since the log of zero or a negative number is undefined.
* The bottom-left and bottom-middle panels test **order**: running mean and
  autocorrelation.
* The bottom-right panel tests for a **mixture**. `np.histogram_bin_edges` computes one set
  of edges from all the data, so both groups are histogrammed on the same bins.
* `.values` turns a pandas column into a plain numpy array, so `x[group == 0]` selects by
  position.

Look at the bottom row. The running mean drifts steadily downward and the autocorrelation
sits above zero at every lag — in a table that shouldn't have any order at all. Check it:
`heart.csv` is **sorted by the label**, all 165 disease patients first, then the 138
without. The order panels have detected the mixture through the way the file was saved.

Two lessons. Order-based panels only measure something physical when row order means
something (time, sequence of measurement) — otherwise they measure how the file was
written, which is worth knowing too. And if you ever split such a file into training and
test sets by taking the first and last rows, this is exactly the trap you'd fall into.

## Diagnose it yourself

That is the whole toolkit: a histogram against the normal it implies, a Q–Q plot, a log, a
running mean, an autocorrelation, and a split by a covariate — six panels, one function.

Putting it to work is **Lab 6**, in the lab session. Five datasets arrive with no labels and
no generating code. Four of them were made by exactly one of the mechanisms in this part — a
mixture, correlated samples, a heavy tail, or a product — and one is a plain normal sample
with nothing wrong with it at all. The job is to say which is which, including which one is
fine, and to name the panel that gave each one away. The lab notebook carries its own copy
of the helpers built above, so it runs on its own.

# Part 3 — Basic Data Manipulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### Workplace Fatalities by State Data

Lets look at some workforce data... unfortunately the source site is no longer avaliable.

The last plot type is the one most readers already know: the bar plot, and with it the pie chart.
These are the plots of school and of reports, and they suit a specific kind of data. This table is
the third level of data from the start of the chapter — no row here is an observation. Each one is
the result of an analysis of hundreds or thousands of individual cases, every workplace death in a
state with its circumstances, reduced to a count, a rate and a few other numbers. A distribution
does not go in a table, but a summary statistic does, and a bar plot is the readable form of such a
table. So bar plots do not appear where the histograms did, at the start of an analysis; they appear
at the end, displaying numbers extracted from the random variables rather than the random variables
themselves. The question this data was assembled to answer is whether states that inspect workplaces
more often have fewer workplace deaths.

The routine is the same as for the heart data, and in the same order: find out how large the file is,
then look at what is in it. Size comes first because it decides what is even possible — a few
kilobytes can be printed and read by eye, and a few gigabytes cannot. `ls -lh` prints the size in
human-readable units; plain `ls` gives only the names.

In [ ]:
!ls -lh

In [ ]:
!head 2012_Workplace_Fatalities_by_State.csv 

This one is a CSV, but not a clean one: some fields clearly hold more than one value. That is not a
reason to write a reader by hand. Try the standard one first and look at what comes back — it parses,
and the repair needed turns out to be confined to a single column.

In [ ]:
df=pd.read_csv("2012_Workplace_Fatalities_by_State.csv")

In [ ]:
df

Two things to take from the loaded table. The first column holds a state name and a pair of
coordinates run together in one field, which is the repair this file needs. And the rows do not all
describe states: the last four are blank or hold national totals and averages. Those rows would
corrupt any maximum, any average and any bar plot computed over a column, which is why everything
below works on rows 0 through 49. The second thing is why a plot is needed at all — fifty numbers in
a column cannot be compared by reading them.

In [ ]:
df.columns[1]

In [ ]:
# To access a column
df[df.columns[1]][:50]

Finding the largest value takes two steps, and the second is the one that matters. The maximum of
the column is a number; the position of that maximum is what identifies the row, and the position is
then used to look up the state's name in the first column. The same pattern — find a position, use
it to index another column — is how any "which row" question gets answered.

Lets find the state with the max number of fatalities:

In [ ]:
# Max of a column
# Note that the last 4 rows are not states
np.max(df[df.columns[1]][0:50])

In [ ]:
# What was the index of the max?
np.argmax(df[df.columns[1]][0:50])

In [ ]:
# What was the state?
df[df.columns[0]][np.argmax(df[df.columns[1]][0:50])]

The answer is Texas, and the answer is useless. The column counts fatalities, and a state with more
workers will record more of them for no other reason, so the largest count simply identifies one of
the largest states. The comparable quantity is a rate — fatalities per worker — which is the same
normalization argument the histograms needed: raw counts cannot be compared across samples of
different size.

The state names aren't quite right... lets look:

In [ ]:
# State names:
df[df.columns[0]][0:50]

The State column is not one field but three, pushed into a single string: a name, a newline, and a
pair of coordinates in parentheses. Nothing can be done with it in that form — it will not sort,
match or label an axis. The repair is the obvious one: split on the newline, keep the first part as
the name, and parse the second part — latitude first, then longitude — into two columns of its own. Work of this
kind is an unglamorous and large share of any real analysis.

In [ ]:
# Clean it up
state_names=list(map(lambda x: x.split("\n")[0],df[df.columns[0]][0:50]))
state_names

In [ ]:
latitude=list(map(lambda x: float(x.split("\n")[1].split(",")[0][1:]),df[df.columns[0]][0:50]))
longitude=list(map(lambda x: float(x.split("\n")[1].split(",")[1][:-1]),df[df.columns[0]][0:50]))

coordinates=list(zip(latitude,longitude))
coordinates

In [ ]:
df.shape

In [ ]:
df["State"]= state_names+[""]*3+["Total/Average"]
df["Longitude"]=longitude+[0.]*4
df["Latitude"]=latitude+[0.]*4

In [ ]:
df

With the table repaired, each bar graph below plots one column against the states: fifty bars, one
per state, with the state names as tick labels. The names are long enough that they have to be rotated
vertically to fit.

In [ ]:
fig,ax= plt.subplots(1,1,figsize=(10,10))

ax.bar(range(50),df[df.columns[1]][0:50])
ax.set_xticks(range(50))
_=ax.set_xticklabels(state_names, rotation='vertical', fontsize=10)
_=ax.set_ylabel(df.columns[1])

This plot is mostly a ranking of states by size. The states with the most workplace deaths are the
states with the most workers, so the tall bars are Texas and California, and the plot would look much
the same for a count of almost anything. What the bar plot does do is its job: fifty numbers that
could not be compared as a column of text become two obvious outliers at a glance. The plot is
readable and the quantity it plots is still the wrong one.

In [ ]:
fig,ax= plt.subplots(1,1,figsize=(10,10))

ax.bar(range(50),df[df.columns[2]][0:50])
ax.set_xticks(range(50))
_=ax.set_xticklabels(state_names, rotation='vertical', fontsize=10)
_=ax.set_ylabel(df.columns[2])

Switching to the rate column changes the ranking completely. The large states fall back towards the
middle and the highest rates belong to small states, North Dakota most of all. That is
worth a second thought rather than a conclusion: a rate computed over a small workforce is built from
few deaths, so it carries a large counting uncertainty, and a state can top this chart on a handful
of cases. The same $\sqrt{n}$ argument used on histogram bins applies to every bar here. This file supplies the
rate itself, so nothing had to be computed; where a dataset does not, working the comparable
quantity out from the columns on hand is feature engineering, and that is most of the work at this
level of analysis. the bar plot stays the right display, because what is being compared is still
one number per state.

In [ ]:
fig,ax= plt.subplots(1,1,figsize=(10,10))

ax.bar(range(50),df[df.columns[9]][0:50])
ax.set_xticks(range(50))
_=ax.set_xticklabels(state_names, rotation='vertical', fontsize=10)
_=ax.set_ylabel(df.columns[9])

This column is the mechanism behind the question. More inspectors means more random inspections,
which raises a workplace's chance of being checked, which should raise the pressure to be safe. The
column measures that as an interval: how long, on average, a given workplace waits between
inspections. Read the axis carefully — the units are *years*, and even the best state is around
thirty, while the worst is above five hundred. Every state has far fewer inspectors than it has
workplaces, so an inspection is a rare event everywhere, and the states differ only in how rare.

In [ ]:
plt.scatter(df[df.columns[2]][0:50],df[df.columns[9]][0:50])

The hypothesis makes a prediction about this plot: longer intervals between inspections should go
with higher fatality rates, so the points should trend upwards from left to right. They do not.
The one state waiting over five hundred years between inspections sits in the middle of the
fatality-rate range, while the two highest rates belong to states waiting about a hundred. Whatever is driving the differences between states, it is not
visible here as a relationship with inspection frequency — which is a result, not a failed plot.

In [ ]:
fig,ax= plt.subplots(1,1,figsize=(10,10))

ax.bar(range(50),df[df.columns[2]][0:50]/df[df.columns[9]][0:50])
ax.set_xticks(range(50))
_=ax.set_xticklabels(state_names, rotation='vertical', fontsize=10)
_=ax.set_ylabel(df.columns[2]+"/"+df.columns[9])

Dividing one column by the other asks the same question in different coordinates, and keeps the
states named. A constant ratio across the states is the same statement as a straight line through
the origin in the scatter plot, so a flat bar graph here would be the relationship the hypothesis
predicted, and a bar that stands out marks a state that breaks the pattern. This is as far as
looking gets. Putting a number on how strong a relationship is, and on how likely it is to be an
accident of fifty data points, is the subject of a statistics course.

Two habits hold this chapter together. The first is that exploring a dataset is a loop, not a
checklist: histogram something, notice a feature, select the rows in it, look at their other
columns, find nothing, go back. Most steps lead nowhere, and the result of the walk is a feel for
what the dataset contains. The second is that every plot is made for a reader, and the first reader
is the person making it. A plot made with no question behind it usually turns out to say nothing,
and noticing that is itself informative.